# Developmental Interpretability: Watching Structure Emerge

*Train a transformer and watch circuits form in real time*

Most interpretability research studies trained models -- frozen snapshots. But how do circuits *form*? When do induction heads appear? When does superposition kick in?

In this notebook we train a small transformer from scratch and track interpretability metrics across training. Instead of studying anatomy, we're studying **embryology** -- watching the development of neural circuits as they come into existence.

**What we'll track:**
1. **Induction head formation** -- when do heads learn to copy patterns?
2. **Direct logit attribution** -- when do heads start contributing to predictions?
3. **Attention entropy** -- when do heads transition from uniform to structured?
4. **Superposition onset** -- when does the model start packing more features than dimensions?

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from copy import deepcopy
import math

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cpu')
print(f"Device: {device}")
print("Ready to watch circuits emerge.")

## The Model and Task

We train a **2-layer, 4-head transformer** with `d_model=128` on a synthetic next-token prediction task.

The data consists of sequences with **repeating patterns** like:
```
[A B C A B C A B ?]  -->  answer: C
[D E D E D ?]        -->  answer: E  
[F G H I F G H ?]   -->  answer: I
```

Solving this task requires:
1. **Bigram statistics** (early training) -- learn which tokens tend to follow which
2. **Copying / induction** (mid training) -- attend to the previous occurrence of the current token, then copy what came after it
3. **Robust pattern completion** (late training) -- handle varying pattern lengths and vocabulary

This is exactly the setting where **induction heads** should emerge -- the canonical example of a circuit forming during training.

In [ ]:
# ============================================================
# Minimal Transformer (self-contained, ~75 lines)
# ============================================================

class Attention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x, return_attn=False):
        B, T, C = x.shape
        q = self.W_Q(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.W_K(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.W_V(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores.masked_fill_(mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn = F.softmax(scores, dim=-1)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        out = self.W_O(out)
        if return_attn:
            return out, attn
        return out

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.attn = Attention(d_model, n_heads)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )
    
    def forward(self, x, return_attn=False):
        if return_attn:
            attn_out, attn_weights = self.attn(self.ln1(x), return_attn=True)
            x = x + attn_out
            x = x + self.mlp(self.ln2(x))
            return x, attn_weights
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class MiniTransformer(nn.Module):
    def __init__(self, vocab_size=64, d_model=128, n_heads=4, n_layers=2, max_len=128):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.unembed = nn.Linear(d_model, vocab_size, bias=False)
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_layers = n_layers
    
    def forward(self, x, return_all=False):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        h = self.tok_emb(x) + self.pos_emb(pos)
        
        attn_maps = []
        residuals = [h.clone()]
        for block in self.blocks:
            if return_all:
                h, attn = block(h, return_attn=True)
                attn_maps.append(attn)
                residuals.append(h.clone())
            else:
                h = block(h)
        
        h = self.ln_f(h)
        logits = self.unembed(h)
        if return_all:
            return logits, attn_maps, residuals
        return logits

# ============================================================
# Data generation: repeating pattern sequences
# ============================================================

def generate_batch(batch_size=64, seq_len=32, vocab_size=64, min_pattern=2, max_pattern=6):
    """Generate sequences with repeating patterns: [A B C A B C A B C ...]"""
    sequences = []
    for _ in range(batch_size):
        pattern_len = np.random.randint(min_pattern, max_pattern + 1)
        # Use tokens 1..vocab_size-1 (reserve 0)
        pattern = torch.randint(1, vocab_size, (pattern_len,))
        # Repeat pattern to fill sequence
        repeats = (seq_len + 1) // pattern_len + 1
        full = pattern.repeat(repeats)[:seq_len + 1]  # +1 for target
        sequences.append(full)
    batch = torch.stack(sequences)
    return batch[:, :-1], batch[:, 1:]  # input, target

# Quick test
model = MiniTransformer()
x, y = generate_batch(batch_size=4, seq_len=16)
logits = model(x)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Input shape: {x.shape}, Output shape: {logits.shape}")
print(f"Example pattern: {x[0].tolist()}")

In [ ]:
# ============================================================
# Training with checkpoint saving
# ============================================================

torch.manual_seed(42)
model = MiniTransformer(vocab_size=64, d_model=128, n_heads=4, n_layers=2)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

n_steps = 2000
checkpoint_every = 100
checkpoints = {}  # step -> state_dict
losses = []

# Save initial checkpoint (step 0 = untrained)
checkpoints[0] = deepcopy(model.state_dict())

print(f"Training for {n_steps} steps...")
for step in range(1, n_steps + 1):
    x, y = generate_batch(batch_size=64, seq_len=32, vocab_size=64)
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, 64), y.view(-1))
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if step % checkpoint_every == 0:
        checkpoints[step] = deepcopy(model.state_dict())
        if step % 500 == 0:
            print(f"  Step {step:4d} | Loss: {loss.item():.4f}")

print(f"\nDone! Final loss: {losses[-1]:.4f}")
print(f"Saved {len(checkpoints)} checkpoints: {sorted(checkpoints.keys())}")

## Metric 1: Induction Head Detection

An **induction head** is an attention head that implements the following algorithm:

> To predict what comes after token `X`, find the *previous* occurrence of `X` in the sequence, then attend to the token that *followed* it there.

For a repeating sequence `[A B C D A B C D]`, an induction head at position 6 (second `C`) should attend strongly to position 3 (the `D` that followed the first `C`).

We measure the **induction score** for each head: the average attention weight placed on the "induction-expected" position across all positions in a repeating sequence. A score near 1.0 means the head is a strong induction head.

In [ ]:
# ============================================================
# Induction head scoring across training
# ============================================================

def compute_induction_scores(state_dict, vocab_size=64, d_model=128, n_heads=4, n_layers=2):
    """Compute induction score for each head at a given checkpoint."""
    m = MiniTransformer(vocab_size, d_model, n_heads, n_layers)
    m.load_state_dict(state_dict)
    m.eval()
    
    # Create a repeating sequence: pattern_len=4, repeated
    pattern = torch.tensor([10, 20, 30, 40])
    seq = pattern.repeat(8)[:32]  # [10,20,30,40,10,20,30,40,...]
    x = seq.unsqueeze(0)  # (1, 32)
    
    with torch.no_grad():
        _, attn_maps, _ = m(x, return_all=True)
    
    scores = {}  # (layer, head) -> induction_score
    pattern_len = 4
    
    for layer_idx, attn in enumerate(attn_maps):
        # attn shape: (1, n_heads, seq_len, seq_len)
        attn = attn[0]  # (n_heads, seq_len, seq_len)
        for head_idx in range(n_heads):
            head_attn = attn[head_idx]  # (seq_len, seq_len)
            induction_scores = []
            # For positions in the second repetition onwards
            for pos in range(pattern_len, len(seq)):
                # Current token at 'pos'
                # Previous occurrence of same token: pos - pattern_len
                # Induction target: (pos - pattern_len) + 1
                prev_occ = pos - pattern_len
                induction_target = prev_occ + 1
                if induction_target < pos:  # must be in causal past
                    induction_scores.append(head_attn[pos, induction_target].item())
            scores[(layer_idx, head_idx)] = np.mean(induction_scores) if induction_scores else 0.0
    
    return scores

# Compute across all checkpoints
print("Computing induction scores across training...")
steps_sorted = sorted(checkpoints.keys())
all_induction_scores = {}  # (layer, head) -> list of scores over steps

for step in steps_sorted:
    scores = compute_induction_scores(checkpoints[step])
    for key, val in scores.items():
        if key not in all_induction_scores:
            all_induction_scores[key] = []
        all_induction_scores[key].append(val)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.tab10(np.linspace(0, 1, 8))
for i, ((layer, head), scores_list) in enumerate(sorted(all_induction_scores.items())):
    label = f"L{layer}.H{head}"
    ax.plot(steps_sorted, scores_list, label=label, color=colors[i], linewidth=2)

ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Induction Score', fontsize=12)
ax.set_title('Induction Head Formation During Training', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_ylim(-0.05, 1.05)
ax.axhline(y=0.066, color='gray', linestyle='--', alpha=0.5, label='chance (~0.066)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Identify strongest induction head
final_scores = {k: v[-1] for k, v in all_induction_scores.items()}
best_head = max(final_scores, key=final_scores.get)
print(f"\nStrongest induction head: L{best_head[0]}.H{best_head[1]} (score: {final_scores[best_head]:.3f})")

## Metric 2: Loss Composition -- When Do Heads Start Helping?

Each attention head produces an output that gets added to the residual stream. We can measure each head's **direct logit attribution** -- how much does projecting this head's output through the unembedding matrix `W_U` contribute to the correct next-token logit?

Formally: `DLA(head) = head_output @ W_U[correct_token, :]`

Note: this approximation ignores the final LayerNorm (`ln_f`), which rescales the residual stream before unembedding. The true logit contribution depends on the LayerNorm gain and centering at each position, so DLA is a first-order estimate rather than an exact decomposition.

Early in training, all heads contribute roughly zero. As training progresses, specific heads suddenly "turn on" and start contributing meaningfully to predictions.

In [ ]:
# ============================================================
# Direct Logit Attribution across training
# ============================================================

def compute_head_dla(state_dict, vocab_size=64, d_model=128, n_heads=4, n_layers=2):
    """Compute direct logit attribution for each head."""
    m = MiniTransformer(vocab_size, d_model, n_heads, n_layers)
    m.load_state_dict(state_dict)
    m.eval()
    
    # Generate a batch of repeating sequences
    torch.manual_seed(99)  # fixed eval data
    x, y = generate_batch(batch_size=32, seq_len=32, vocab_size=64)
    
    with torch.no_grad():
        B, T = x.shape
        pos = torch.arange(T).unsqueeze(0)
        h = m.tok_emb(x) + m.pos_emb(pos)
        
        W_U = m.unembed.weight  # (vocab_size, d_model)
        head_dlas = {}  # (layer, head) -> mean DLA
        
        for layer_idx, block in enumerate(m.blocks):
            # Get attention output per head (before W_O combines them)
            ln_out = block.ln1(h)
            attn = block.attn
            d_head = attn.d_head
            
            q = attn.W_Q(ln_out).view(B, T, n_heads, d_head).transpose(1, 2)
            k = attn.W_K(ln_out).view(B, T, n_heads, d_head).transpose(1, 2)
            v = attn.W_V(ln_out).view(B, T, n_heads, d_head).transpose(1, 2)
            
            scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_head)
            causal_mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
            scores.masked_fill_(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
            attn_weights = F.softmax(scores, dim=-1)
            
            head_outputs = attn_weights @ v  # (B, n_heads, T, d_head)
            
            # Project each head through W_O and then W_U to get logit contribution
            W_O = attn.W_O.weight  # (d_model, d_model)
            
            for head_idx in range(n_heads):
                # head_out: (B, T, d_head)
                head_out = head_outputs[:, head_idx]  # (B, T, d_head)
                
                # W_O maps from concat of all heads -> d_model
                # For head_idx, the relevant slice of W_O
                start = head_idx * d_head
                end = (head_idx + 1) * d_head
                W_O_head = W_O[:, start:end]  # (d_model, d_head)
                
                # Project: head_out @ W_O_head^T @ W_U^T -> logits
                projected = head_out @ W_O_head.T  # (B, T, d_model)
                logit_contrib = projected @ W_U.T  # (B, T, vocab_size)
                
                # Get contribution to correct token logit
                correct_logits = logit_contrib.gather(2, y.unsqueeze(-1)).squeeze(-1)  # (B, T)
                head_dlas[(layer_idx, head_idx)] = correct_logits.mean().item()
            
            # Full forward through block for next layer
            h = h + block.attn(block.ln1(h))
            h = h + block.mlp(block.ln2(h))
    
    return head_dlas

# Compute across checkpoints
print("Computing direct logit attribution across training...")
all_dlas = {}  # (layer, head) -> list of DLA values

for step in steps_sorted:
    dlas = compute_head_dla(checkpoints[step])
    for key, val in dlas.items():
        if key not in all_dlas:
            all_dlas[key] = []
        all_dlas[key].append(val)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
for i, ((layer, head), dla_list) in enumerate(sorted(all_dlas.items())):
    label = f"L{layer}.H{head}"
    ax.plot(steps_sorted, dla_list, label=label, color=colors[i], linewidth=2)

ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Direct Logit Attribution', fontsize=12)
ax.set_title('Head Contributions to Correct Predictions Over Training', fontsize=14, fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Heads that contribute most to final predictions:")
final_dlas = {k: v[-1] for k, v in all_dlas.items()}
for k, v in sorted(final_dlas.items(), key=lambda x: -x[1]):
    print(f"  L{k[0]}.H{k[1]}: {v:.4f}")

## Metric 3: Attention Entropy -- From Uniform to Structured

An untrained transformer has roughly **uniform attention** -- each position attends equally to all previous positions. This corresponds to **high entropy**.

As training progresses, heads **specialize**: some become position heads, some become induction heads, some attend to specific token types. This specialization shows up as **decreasing entropy**.

We compute: `H(head) = -sum(p * log(p))` where `p` is the attention distribution, averaged over positions and sequences.

In [ ]:
# ============================================================
# Attention entropy across training
# ============================================================

def compute_attn_entropy(state_dict, vocab_size=64, d_model=128, n_heads=4, n_layers=2):
    """Compute mean attention entropy for each head."""
    m = MiniTransformer(vocab_size, d_model, n_heads, n_layers)
    m.load_state_dict(state_dict)
    m.eval()
    
    torch.manual_seed(99)
    x, _ = generate_batch(batch_size=32, seq_len=32, vocab_size=64)
    
    with torch.no_grad():
        _, attn_maps, _ = m(x, return_all=True)
    
    entropies = {}
    for layer_idx, attn in enumerate(attn_maps):
        # attn: (B, n_heads, T, T)
        for head_idx in range(n_heads):
            head_attn = attn[:, head_idx]  # (B, T, T)
            # Compute entropy, avoiding log(0)
            log_attn = torch.log(head_attn + 1e-10)
            entropy = -(head_attn * log_attn).sum(dim=-1)  # (B, T)
            # Average over batch and positions (skip pos 0 which has no choice)
            entropies[(layer_idx, head_idx)] = entropy[:, 1:].mean().item()
    
    return entropies

# Compute across checkpoints
print("Computing attention entropy across training...")
all_entropies = {}

for step in steps_sorted:
    ents = compute_attn_entropy(checkpoints[step])
    for key, val in ents.items():
        if key not in all_entropies:
            all_entropies[key] = []
        all_entropies[key].append(val)

# Compute max possible entropy for reference (uniform over causal positions)
# Position t can attend to t+1 positions -> entropy = log(t+1)
# Average over positions 1..31: mean(log(2), log(3), ..., log(32))
max_entropy = np.mean([np.log(t+1) for t in range(1, 32)])

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
for i, ((layer, head), ent_list) in enumerate(sorted(all_entropies.items())):
    label = f"L{layer}.H{head}"
    ax.plot(steps_sorted, ent_list, label=label, color=colors[i], linewidth=2)

ax.axhline(y=max_entropy, color='gray', linestyle='--', alpha=0.5, label=f'Uniform ({max_entropy:.2f})')
ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Attention Entropy (nats)', fontsize=12)
ax.set_title('Attention Entropy: From Uniform to Structured', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Max possible entropy (uniform): {max_entropy:.3f}")
print(f"Final entropies:")
for k, v in sorted(all_entropies.items()):
    print(f"  L{k[0]}.H{k[1]}: {v[-1]:.3f} (started at {v[0]:.3f})")

## Metric 4: Feature Superposition Onset

Early in training, the model uses **dedicated dimensions** for each feature -- one feature per direction in activation space. As the model learns to represent more concepts than it has dimensions, it begins to **superpose** features, packing multiple features into overlapping directions.

We can detect this by measuring the **effective rank** of the activation matrix:

$$\text{Effective Rank} = \exp\left(-\sum_i \hat{\sigma}_i \log \hat{\sigma}_i\right)$$

where $\hat{\sigma}_i$ are the normalized singular values. This measures how many dimensions are "actively used" by the representations. An increase in effective rank means the model is learning to use more of its available dimensions. Note that this is distinct from superposition: increasing effective rank means utilising more orthogonal directions, whereas superposition means representing *more features than dimensions* by encoding them in overlapping, non-orthogonal directions. A rising effective rank may be a precondition for superposition (the model first spreads out, then packs further), but the two phenomena should not be conflated.

In [ ]:
# ============================================================
# Effective rank of representations across training
# ============================================================

def compute_effective_rank(state_dict, vocab_size=64, d_model=128, n_heads=4, n_layers=2):
    """Compute effective rank of final-layer residual stream activations."""
    m = MiniTransformer(vocab_size, d_model, n_heads, n_layers)
    m.load_state_dict(state_dict)
    m.eval()
    
    # Collect activations over multiple batches
    all_acts = []
    torch.manual_seed(99)
    for _ in range(4):  # 4 batches of 32 = 128 sequences
        x, _ = generate_batch(batch_size=32, seq_len=32, vocab_size=64)
        with torch.no_grad():
            _, _, residuals = m(x, return_all=True)
            # Last residual stream (after final block)
            acts = residuals[-1]  # (B, T, d_model)
            all_acts.append(acts.reshape(-1, d_model))  # flatten B*T
    
    activations = torch.cat(all_acts, dim=0)  # (N, d_model)
    # Center
    activations = activations - activations.mean(dim=0, keepdim=True)
    
    # SVD
    _, S, _ = torch.svd(activations)
    
    # Normalized singular values
    S_norm = S / S.sum()
    S_norm = S_norm[S_norm > 1e-10]  # avoid log(0)
    
    # Effective rank = exp(entropy of normalized singular values)
    entropy = -(S_norm * torch.log(S_norm)).sum().item()
    eff_rank = np.exp(entropy)
    
    # Variance explained by top-K
    S_sq = (S ** 2)
    total_var = S_sq.sum().item()
    top10_var = S_sq[:10].sum().item() / total_var
    top50_var = S_sq[:50].sum().item() / total_var
    
    return eff_rank, top10_var, top50_var, S.numpy()

# Compute at every 5th checkpoint for speed
print("Computing effective rank across training (every 5th checkpoint)...")
rank_steps = steps_sorted[::5]  # every 5th
eff_ranks = []
top10_vars = []
top50_vars = []
all_singular_vals = []

for step in rank_steps:
    er, t10, t50, sv = compute_effective_rank(checkpoints[step])
    eff_ranks.append(er)
    top10_vars.append(t10)
    top50_vars.append(t50)
    all_singular_vals.append(sv)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Effective rank
axes[0].plot(rank_steps, eff_ranks, 'b-', linewidth=2)
axes[0].set_xlabel('Training Step', fontsize=12)
axes[0].set_ylabel('Effective Rank', fontsize=12)
axes[0].set_title('Representation Dimensionality Over Training', fontsize=13, fontweight='bold')
axes[0].axhline(y=128, color='gray', linestyle='--', alpha=0.5, label='Max (d_model=128)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: Variance explained
axes[1].plot(rank_steps, top10_vars, 'r-', linewidth=2, label='Top 10 dims')
axes[1].plot(rank_steps, top50_vars, 'g-', linewidth=2, label='Top 50 dims')
axes[1].set_xlabel('Training Step', fontsize=12)
axes[1].set_ylabel('Fraction of Variance Explained', fontsize=12)
axes[1].set_title('Variance Concentration Over Training', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Effective rank: {eff_ranks[0]:.1f} (init) -> {eff_ranks[-1]:.1f} (final)")
print(f"Top-10 variance: {top10_vars[0]:.3f} (init) -> {top10_vars[-1]:.3f} (final)")

## The Full Picture: Phase Transitions in Training

Now let's bring all four metrics together to see the **developmental trajectory** of our transformer. By plotting everything on the same x-axis (training step), we can identify distinct **phases** of development:

1. **Random phase** (early): High entropy, no induction, low DLA
2. **Bigram phase**: Entropy drops as heads learn simple positional/bigram patterns
3. **Induction phase**: Specific heads snap into induction behavior (phase transition!)
4. **Refinement phase**: Representations become richer, superposition may emerge

In [ ]:
# ============================================================
# Developmental Interpretability Dashboard
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Developmental Interpretability Dashboard', fontsize=16, fontweight='bold', y=1.02)

# --- Panel 1: Loss curve ---
ax = axes[0, 0]
# Smooth the loss for clarity
window = 50
smoothed_loss = np.convolve(losses, np.ones(window)/window, mode='valid')
loss_steps = np.arange(len(smoothed_loss)) + window // 2
ax.plot(loss_steps, smoothed_loss, 'k-', linewidth=2)
ax.set_xlabel('Training Step')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Training Loss', fontweight='bold')
ax.grid(True, alpha=0.3)

# --- Panel 2: Induction scores ---
ax = axes[0, 1]
for i, ((layer, head), scores_list) in enumerate(sorted(all_induction_scores.items())):
    label = f"L{layer}.H{head}"
    ax.plot(steps_sorted, scores_list, label=label, color=colors[i], linewidth=2)
ax.set_xlabel('Training Step')
ax.set_ylabel('Induction Score')
ax.set_title('Induction Head Formation', fontweight='bold')
ax.legend(fontsize=8, loc='upper left')
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)

# --- Panel 3: Attention entropy ---
ax = axes[1, 0]
for i, ((layer, head), ent_list) in enumerate(sorted(all_entropies.items())):
    label = f"L{layer}.H{head}"
    ax.plot(steps_sorted, ent_list, label=label, color=colors[i], linewidth=2)
ax.axhline(y=max_entropy, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Training Step')
ax.set_ylabel('Entropy (nats)')
ax.set_title('Attention Entropy', fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)

# --- Panel 4: Effective rank ---
ax = axes[1, 1]
ax.plot(rank_steps, eff_ranks, 'b-', linewidth=2)
ax.set_xlabel('Training Step')
ax.set_ylabel('Effective Rank')
ax.set_title('Representation Dimensionality', fontweight='bold')
ax.axhline(y=128, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)

# --- Add phase transition markers ---
# Identify approximate transition points from induction scores
# Look for the step where the best induction head crosses 0.3
best_key = max(all_induction_scores.keys(), key=lambda k: all_induction_scores[k][-1])
best_scores = all_induction_scores[best_key]
transition_step = None
for i, (step, score) in enumerate(zip(steps_sorted, best_scores)):
    if score > 0.3:
        transition_step = step
        break

if transition_step:
    for ax in axes.flat:
        ax.axvline(x=transition_step, color='red', linestyle=':', alpha=0.6, linewidth=1.5)
    # Add annotation to first panel
    axes[0, 0].annotate(f'Induction\nphase\n(step {transition_step})',
                        xy=(transition_step, axes[0, 0].get_ylim()[1] * 0.7),
                        fontsize=9, color='red', ha='center',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print("Dashboard complete.")
if transition_step:
    print(f"Induction phase transition detected around step {transition_step}")
print("Red dashed lines mark the approximate onset of induction head behavior.")

## Key Takeaways

**1. Circuits don't form gradually -- they snap into place.** The induction score plots show something remarkable: heads go from near-zero to strong induction behavior over a narrow window of training steps. This is a **phase transition**, not a smooth gradient.

**2. Different heads specialize at different times.** Not all heads develop simultaneously. Some heads may specialize early (e.g., positional attention), while induction heads typically emerge later, building on the earlier heads' representations.

**3. Entropy tells a story of specialization.** The drop from uniform (high-entropy) to structured (low-entropy) attention reveals when heads "decide" what to attend to. Sharp entropy drops often coincide with circuit formation.

**4. Superposition emerges as capacity pressure grows.** The effective rank of representations changes as the model learns to pack more information into its fixed-width residual stream.

**5. Connection to the literature:**
- **Olsson et al. (2022)**, "In-context Learning and Induction Heads" -- first documented the induction head phase transition, showing it coincides with the emergence of in-context learning
- **Neel Nanda et al.** -- studied grokking and phase transitions in algorithmic learning, finding sudden circuit formation
- **Anthropic's work on superposition** -- showed that models learn to represent more features than dimensions as training progresses

**Open question:** Can we predict *which* circuits will form from the architecture and data alone? If we understand developmental interpretability deeply enough, we might be able to predict the structure of trained models before training them -- a powerful tool for alignment and safety.